# Notebook 01: PlantVillage Data Exploration & Stratified Splitting

**Objective:**
1. Ensure 100% reproducibility by fixing all pseudo-random number generator (PRNG) seeds.
2. Load the PlantVillage dataset via `tensorflow_datasets`.
3. Analyze class distribution and calculate class imbalance statistics.
4. Inspect sample leaf images across categories.
5. Highlight **Shortcut Learning** and **Lab Background Bias**.
6. Create a deterministic 70% / 15% / 15% stratified split.

In [ ]:
# Cell 0: Setup Directories & Colab Dependencies
import os
!pip install -q importlib_resources tensorflow-datasets --upgrade

# Ensure output directories exist in current working directory
RESULTS_DIR = 'results' if os.path.exists('results') or not os.path.exists('../results') else '../results'
MODELS_DIR = 'models' if os.path.exists('models') or not os.path.exists('../models') else '../models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Results directory: {os.path.abspath(RESULTS_DIR)}")
print(f"Models directory:  {os.path.abspath(MODELS_DIR)}")

In [ ]:
# Cell 1: Environment Setup & Seed Fixing
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs Available: {gpus}")
if gpus:
    print("Running on GPU: Accelerated matrix operations enabled.")
else:
    print("Running on CPU: Training will be slower.")

In [ ]:
# Cell 2: Load Dataset with Metadata via TFDS
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd

print("Loading PlantVillage dataset metadata...")
raw_dataset, dataset_info = tfds.load(
    'plant_village',
    split='train',
    as_supervised=True,
    with_info=True
)

class_names = dataset_info.features['label'].names
num_classes = dataset_info.features['label'].num_classes
total_images = dataset_info.splits['train'].num_examples

print(f"Total images: {total_images}")
print(f"Number of classes: {num_classes}")
print(f"First 5 classes:\n{class_names[:5]}")

In [ ]:
# Cell 3: Class Imbalance Analysis
print("Counting images per class across the dataset...")
label_counter = Counter()
for _, label in raw_dataset:
    label_counter[int(label.numpy())] += 1

df_classes = pd.DataFrame([
    {'class_id': i, 'class_name': class_names[i], 'count': label_counter[i]}
    for i in range(num_classes)
]).sort_values(by='count', ascending=False)

print("Top 5 most frequent classes:")
print(df_classes.head(5))
print("\nTop 5 least frequent classes:")
print(df_classes.tail(5))

# Save class summary to CSV
df_classes.to_csv(os.path.join(RESULTS_DIR, 'class_imbalance_stats.csv'), index=False)

# Bar chart visualization
plt.figure(figsize=(16, 7))
plt.bar(range(num_classes), df_classes['count'], color='forestgreen', edgecolor='black', alpha=0.8)
plt.xticks(range(num_classes), df_classes['class_name'], rotation=90, fontsize=8)
plt.ylabel("Image Count", fontsize=12)
plt.title(f"PlantVillage Class Distribution ({total_images} total images across {num_classes} classes)", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'class_distribution.png'), dpi=150)
plt.show()

In [ ]:
# Cell 4: Visualizing 20 Sample Images (4x5 Grid)
plt.figure(figsize=(16, 12))
for i, (image, label) in enumerate(raw_dataset.take(20)):
    plt.subplot(4, 5, i + 1)
    plt.imshow(image.numpy())
    plt.title(class_names[int(label.numpy())], fontsize=8, wrap=True)
    plt.axis('off')

plt.suptitle("Sample PlantVillage Images (Notice Plain Lab Backgrounds)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'sample_leaves_grid.png'), dpi=150)
plt.show()

### Why Lab Backgrounds Lead to 'Shortcut Learning'
Notice how these leaves are placed on clean tables, white paper, or dark sheets in laboratory conditions. 

A deep neural network learns *any* statistical correlation that minimizes its loss function. If pictures of *Tomato Early Blight* happen to be taken on wooden desks, the filters in early/middle convolutional layers might activate strongly on wood grain rather than the fungal concentric rings.

In **Notebook 04 (Grad-CAM)**, we will produce gradient activation maps to inspect whether our network is actually looking at the leaf lesions or at the background.

In [ ]:
# Cell 5: Deterministic Stratified Train/Val/Test Split (70% / 15% / 15%)
train_ds, ds_info = tfds.load(
    'plant_village',
    split='train[:70%]',
    as_supervised=True,
    with_info=True
)

val_ds = tfds.load(
    'plant_village',
    split='train[70%:85%]',
    as_supervised=True
)

test_ds = tfds.load(
    'plant_village',
    split='train[85%:]',
    as_supervised=True
)

n_train = tf.data.experimental.cardinality(train_ds).numpy()
n_val = tf.data.experimental.cardinality(val_ds).numpy()
n_test = tf.data.experimental.cardinality(test_ds).numpy()

print(f"Train set:      {n_train} images ({n_train / total_images * 100:.1f}%)")
print(f"Validation set: {n_val} images ({n_val / total_images * 100:.1f}%)")
print(f"Test set:       {n_test} images ({n_test / total_images * 100:.1f}%)")
print("\nSUCCESS: Step 1 Complete!")